In [ ]:

import os
import numpy as np
from cpom.altimetry.datasets.dataset_helper import DatasetHelper
import polars as pl 

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed


In [3]:
this_dataset = DatasetHelper("/home/willisc3/luna/CPOM/archive/SATS/RA/CRY/L2/CRYO-TEMPO/BASELINE-D/LAND_ICE/GREENL/2024",
                            "/home/willisc3/cpom_software2/src/cpom/altimetry/datasets/definitions/cryotempo.yml")

variables = ['basin_id', 'backscatter']

In [16]:
from concurrent.futures import ThreadPoolExecutor, as_completed


In [18]:
pq = pl.scan_parquet("/home/willisc3/luna/CPOM/willisc3/new_archive_test1/north/cs2/CryoTEMPO_baseline_D/2024/*.parquet")

# Build index once
file_index = {
    Path(f).stem: os.path.join(root, f)
    for root, _, files in os.walk(this_dataset.data_dir)
    for f in files
}

# Get source/idx mapping
source_idx = (
    pq.select(["source", "idx"])
    .collect()
    .group_by("source")
    .agg(pl.col("idx").alias("indices"))
    .sort("source")
)

def make_reader(file_index, this_dataset, variables):
    def read_source(row):
        source = row["source"]
        idx = np.sort(np.array(row["indices"]))
        path = file_index.get(source)
        if path is None:
            return None
        with this_dataset.get_file_handle(path) as nc:
            frame_data = {var: this_dataset.get_variable(nc, var, idx) for var in variables}
            frame_data["source"] = np.full(len(idx), source)
            frame_data["idx"] = idx
        return pl.DataFrame(frame_data).lazy()
    return read_source

read_source = make_reader(file_index, this_dataset, variables)
rows = source_idx.iter_rows(named=True)

with ThreadPoolExecutor(max_workers=8) as pool:
    futures = [pool.submit(read_source, row) for row in rows]
    frames = [f.result() for f in as_completed(futures) if f.result() is not None]

enriched = (
    pq.join(
        pl.concat(frames, rechunk=False),
        on=["source", "idx"],
        how="left",
    )
    .collect(streaming=True)
)

: 